# Clock

## Overview 
Later phases join the sources on time, so their timestamps need to agree. If the server isn't on UTC, every join to DS4 is shifted (by an hour in summer on UK time, or nine hours on JST), so this notebook first checks which time zone the server clock is on.

**Question.** DS1-3 carry server-local timestamps with no time zone, while DS4 is epoch ms (UTC). Is the server clock on UTC, so joins to DS4 need no offset?

**Method.** The server names each gzip file from its local clock, and `gzip` stamps the header with Unix time, which doesn't have a time zone (section 1). If the two agree, the server is on UTC (section 2). DS1, DS3 and the run map are then connected back to those files (section 3), and DS4's raw time column is checked last (section 4). Results are in [Phase 01, A1](../../docs/phases/01_eda.md#a1-clock).

In [1]:
import datetime as dt
import re
import struct

import numpy as np
import pandas as pd

from qpu_drift.derived.run_map import load_run_map
from qpu_drift.paths import RAW
from qpu_drift.sources.benchmarking import load_runs_l2
from qpu_drift.sources.measurements import load_measurements


def season(t: pd.Series) -> np.ndarray:
    """Rough BST/GMT split: April to October counts as summer."""
    return np.where(t.dt.month.between(4, 10), "summer", "winter")

## 1. Where the times come from

I have the local code for `oqc-maac` and `oqc-calibration-routines` (in `~/git`) - so I search for the lines that set each timestamp or file name, and the gzip writes. The clones are at their current head, so this shows how the stamps are made today - section 2 checks the data itself.

In [2]:
import subprocess
from pathlib import Path

# Lines that set a timestamp, name a file or write a gzip
PATTERN = (
    r"(time_start|time_end) = datetime\.now\(\)"
    r"|d = (datetime\.now\(\)|result\.time_end)"
    r"|def get_(result|calibration)_filename"
    r"|def get_output_info_path|dt = datetime\.datetime\.now\(\)"
    r'|gzip\.open\(filepath, "w"\)'
    r'|"created_at": str\(datetime'
)

# Files to search in each local clone
SOURCES = {
    "oqc-maac": [
        "maac/measurement/base.py",
        "maac/measurement_environment/main.py",
        "maac/measurement_environment/database.py",
        "maac/measurement_environment/encoder.py",
    ],
    "oqc-calibration-routines": [
        "calibration_routines/scripts/benchmarking.py",
        "calibration_routines/utilities/util.py",
    ],
}

for repo, files in SOURCES.items():
    out = subprocess.run(
        ["git", "-C", Path.home() / "git" / repo, "grep", "-n", "-E", PATTERN, "--", *files],
        capture_output=True,
        text=True,
        check=True,
    ).stdout
    print(f"{repo}:\n{out}")

oqc-maac:
maac/measurement/base.py:82:            self.time_start = datetime.now()
maac/measurement_environment/database.py:91:    def get_result_filename(self, result):
maac/measurement_environment/database.py:92:        d = result.time_end
maac/measurement_environment/database.py:135:                result_db.time_end = result.time_end
maac/measurement_environment/encoder.py:59:        with gzip.open(filepath, "w") as fout:
maac/measurement_environment/encoder.py:89:        with gzip.open(filepath, "w") as fout:
maac/measurement_environment/main.py:445:        result.time_end = datetime.now()
maac/measurement_environment/main.py:522:    def get_calibration_filename(self, use_compression=True):
maac/measurement_environment/main.py:527:        d = datetime.now()
maac/measurement_environment/main.py:890:        result.time_end = datetime.now()

oqc-calibration-routines:
calibration_routines/scripts/benchmarking.py:52:            "created_at": str(datetime.datetime.now()),
calibration_ro

Every stamp comes from `datetime.now()` on the server: DS1's start and end times, the DS2 snapshot and `output_info` file names, and DS3's `created_at`. The DS1 result file name is built from `time_end`, and maac writes its files with `gzip.open` - so the gzip header gives us a second, time-zone-free clock to check the names against.

## 2. The gzip headers put the server on UTC

This is the main test. For every gzip file we take the Unix time from the header, read the file name as if it were UTC, and look at the difference in seconds:

$$\text{offset} = t_{\text{gzip header}} - t_{\text{file name (UTC)}}$$

If the server is on UTC this should sit near zero. On UK time it'd be about $-3{,}600$ s in summer, and on JST about $-32{,}400$ s all year round. We split by season too (April to October as summer - a rough BST split) so a daylight-saving shift would show up.

In [3]:
# Offset of every gzip file: header time minus the time in its name
rows = []
for src in ("hardware_models", "raw_ramsey"):
    for f in (RAW / src).rglob("*.gz"):
        with f.open("rb") as fh:
            mtime = struct.unpack("<I", fh.read(8)[4:8])[0]  # gzip MTIME, Unix epoch

        # Time in the file name, read as UTC
        stamp = re.search(r"\d{4}/\d\d/\d\d/\d\d-\d\d-\d\d", f.as_posix())[0]
        name = pd.Timestamp(dt.datetime.strptime(stamp, "%Y/%m/%d/%H-%M-%S"), tz="UTC")
        rows.append((src, name, mtime - name.timestamp()))

# Offsets as a table, with the season of each file name
gz = pd.DataFrame(rows, columns=["source", "name_utc", "offset_s"])
gz["season"] = season(gz["name_utc"])
print("files per source:", gz["source"].value_counts().to_dict(), "| total:", len(gz))

# Number of files and offset range per source and season
gz.groupby(["source", "season"])["offset_s"].agg(
    files="size", min_offset_s="min", median_offset_s="median", max_offset_s="max"
)

files per source: {'hardware_models': 4603, 'raw_ramsey': 2501} | total: 7104


files  min_offset_s  median_offset_s  max_offset_s
source          season                                                    
hardware_models summer   2831           0.0              1.0           5.0
                winter   1772           0.0              1.0           2.0
raw_ramsey      summer   1436           0.0              0.0          34.0
                winter   1065           0.0              0.0           1.0

Every one of the 7,104 headers is 0 to 34 s after its file name, with a median of 0 to 1 s in summer and winter alike. There's no hour's step in summer and no nine-hour shift, so the server is on UTC.

## 3. DS1, DS3 and the run map share that clock

Now the file names are pinned to UTC, we can connect the other sources back to them.

**DS1.** maac builds each result file name from the row's `time_end` (section 1), so a Ramsey row's `time_end` should match the stamp in its `relative_filepath`. That's true by design - the point is it links DS1 to the files checked in section 2.

**DS3.** If DS3 shares DS1's clock, each run's `created_at` should land just after its last DS1 `RandomizedBenchmarking2q` row. So the lag is

$$\text{lag}_{\text{DS3}} = t_{\text{created\_at}} - t_{\text{time\_end}}^{\text{last RB2q}}$$

An hour's offset either way would push it well off zero - to about 3,600 s, or to about 23 h (the backward join would pick up the previous day's RB2q instead). We bin all the lags first, then take the median of those under 10 min per season. A shared clock moves both stamps together at a DST switch, so the season split can't catch DST here - it just shows whether the two drift apart over the year.

In [4]:
# DS1: Ramsey time_end against the time stamp in the raw file path
ds1 = load_measurements()
ram = ds1[ds1["measurement_type"] == "Ramsey"].copy()
ram["stamp"] = pd.to_datetime(
    ram["relative_filepath"].str.extract(r"(\d{4}/\d\d/\d\d/\d\d-\d\d-\d\d)")[0],
    format="%Y/%m/%d/%H-%M-%S",
)
print(
    f"Ramsey time_end = file stamp: {(ram['time_end'].dt.floor('s') == ram['stamp']).sum():,}"
    f" of {len(ram):,}"
)

# DS3: lag from the last DS1 RB2q time_end before each run
is_rb2q = ds1["measurement_type"] == "RandomizedBenchmarking2q"
rb2q = ds1.loc[is_rb2q, ["time_end"]].sort_values("time_end")
ds3 = load_runs_l2()[["created_at"]].sort_values("created_at")
ds3 = ds3[ds3["created_at"] >= rb2q["time_end"].min()]

# Lag in seconds, binned
lag = pd.merge_asof(ds3, rb2q, left_on="created_at", right_on="time_end", direction="backward")
lag["lag_s"] = (lag["created_at"] - lag["time_end"]).dt.total_seconds()
print(f"\nDS3 runs after the first RB2q: {len(lag)}; lag bins (s):")
print(
    pd.cut(lag["lag_s"], [0, 600, 3000, 4200, 86400, np.inf], right=False)
    .value_counts(sort=False)
    .to_string()
)

# Lags of 10 min or more: runs after DS1 ends, and the rest
far = lag[lag["lag_s"] >= 600]
after = far["created_at"] > ds1["time_end"].max()
print(
    f"\nlag of 10 min or more: {len(far)}; "
    f"after DS1 ends ({ds1['time_end'].max():%Y-%m-%d %H:%M}): {int(after.sum())}"
)

# The rest, with the benchmarking run that started just before
bench = (
    load_run_map(routine="benchmarking")[["start", "n_result_ids"]]
    .sort_values("start")
    .astype({"start": far["created_at"].dtype})
)
print(pd.merge_asof(far[~after], bench, left_on="created_at", right_on="start").to_string(index=False))

# Median lag of the runs under 10 min, per season
near = lag[lag["lag_s"] < 600]
(
    near.groupby(season(near["created_at"]))["lag_s"]
    .agg(ds3_runs="size", median_lag_s="median")
    .round(2)
)

Ramsey time_end = file stamp: 2,525 of 2,525

DS3 runs after the first RB2q: 580; lag bins (s):
lag_s
[0.0, 600.0)         563
[600.0, 3000.0)        0
[3000.0, 4200.0)       0
[4200.0, 86400.0)      1
[86400.0, inf)        16

lag of 10 min or more: 17; after DS1 ends (2026-08-18 09:13): 16
                created_at                   time_end        lag_s               start  n_result_ids
2026-08-10 13:56:30.836100 2026-08-09 15:57:30.079168 79140.756932 2026-08-10 13:53:57             0


,ds3_runs,median_lag_s
summer,325,1.68
winter,238,1.82


All 2,525 Ramsey `time_end` values match their file stamp to the second, so DS1 is on the file clock. 563 of the 580 DS3 runs land within 10 min of the last RB2q row (median 1.68 s in summer, 1.82 s in winter), and none sits anywhere near 3,600 s. The other 17 aren't clock errors: 16 come after DS1 ends on 2026-08-18, and the run of 2026-08-10 13:56 follows a benchmarking run that wrote no DS1 rows, so the join reaches back to an earlier RB2q.

**Run map.** Each run's start comes from its `output_info` file name, so we take the lag from that to the first DS1 row it validly links, per routine and season:

$$\text{lag}_{\text{run map}} = t_{\text{first linked DS1 row}} - t_{\text{start}}$$

In [5]:
from qpu_drift.derived.run_map import load_run_results

# Run map: lag from each run's start to its first valid DS1 row
res = load_run_results()

# Keep the valid links and join the DS1 start time
res = res[res["valid"]].merge(ds1[["id", "time_start"]], left_on="result_id", right_on="id")

# First DS1 row per run, then the lag in minutes
first = res.groupby(["routine", "start"])["time_start"].min().reset_index()
first["lag_min"] = (first["time_start"] - first["start"]).dt.total_seconds() / 60

# Runs whose first DS1 row comes before the run start
neg = first[first["lag_min"] < 0].round({"lag_min": 2})
print("negative lags:\n", neg.to_string(index=False), "\n")

# Lag per routine and season
first.groupby(["routine", season(first["start"])])["lag_min"].agg(
    runs="size", min_lag_min="min", median_lag_min="median", max_lag_min="max"
).round(2)

negative lags:
      routine               start                 time_start  lag_min
benchmarking 2025-08-22 08:26:52 2025-08-21 20:11:39.562803  -735.21 



runs  min_lag_min  median_lag_min  max_lag_min
routine                                                                 
benchmarking      summer   348      -735.21            0.02         0.03
                  winter   256         0.01            0.01         0.03
calibrate_1q      summer   714         0.00            0.01         0.31
                  winter   532         0.00            0.01         0.03
calibrate_2q      summer  1425         0.00            0.02         1.29
                  winter   529         0.01            0.18         1.12
mixer_calibration summer   374         0.02            0.03         0.19
                  winter   257         0.02            0.03         0.10

Every routine's first DS1 row comes a median of 0.01 to 0.18 min after the run starts, and never more than 1.29 min, in both seasons. The one negative lag is the `benchmarking` file of 2025-08-22 08:26:52, whose first DS1 row is from the evening before: that file was written again the morning after its run, so it's the file name that's late, not the clock ([datasheet 3, section 6](../../docs/datasheets/03_benchmarking_files.md#6-preprocessing-and-cleaning); also [B3a section 1](../01_profiling/b3a_ds1_spine.ipynb)).

## 4. DS4 is epoch ms in UTC

Lastly, a quick look at DS4 on its own. We print the first three raw `Time` values from the Grafana export, the same values parsed as epoch ms in UTC, and the first three times `load_fridge` gives back.

In [6]:
from qpu_drift.sources.fridge import load_fridge

# First three raw Time values, parsed as epoch ms in UTC, against load_fridge
raw = pd.read_csv(RAW / "fridge_lx2_grafana_20260903.csv", usecols=["Time"], nrows=3)
print("raw Time:", raw["Time"].tolist())
print("as epoch ms, UTC:", pd.to_datetime(raw["Time"], unit="ms", utc=True).tolist())
print("load_fridge index:", load_fridge().index[:3].tolist())

raw Time: [1754049600000, 1754071200000, 1754092800000]
as epoch ms, UTC: [Timestamp('2025-08-01 12:00:00+0000', tz='UTC'), Timestamp('2025-08-01 18:00:00+0000', tz='UTC'), Timestamp('2025-08-02 00:00:00+0000', tz='UTC')]
load_fridge index: [Timestamp('2025-08-01 12:00:00+0000', tz='UTC'), Timestamp('2025-08-01 18:00:00+0000', tz='UTC'), Timestamp('2025-08-02 00:00:00+0000', tz='UTC')]


The raw values parse as epoch ms to 6-hourly UTC times from 2025-08-01 12:00, and `load_fridge` gives back the same times in UTC. With the server on UTC too, DS4 joins to the other sources with no offset.

## Conclusion

| Result | Status |
|---|---|
| The server is on UTC: all 7,104 gzip headers match their file names read as UTC, in both seasons | Confirmed |
| DS1 and DS3 share that clock: Ramsey `time_end` equals its file stamp, and DS3 runs end about 2 s after their last RB2q row | Confirmed |
| So do run-map starts: each run's first DS1 row follows its start within about a minute, apart from one benchmarking file written again the next morning | Confirmed |
| DS4 is epoch ms in UTC, so it joins the other sources with no offset | Confirmed |